# Enveda CASMI 2026 — submission notebook: baseline (Ridge) and deep MLP
Predicts up to 25 candidate structures per `molecule_id` and writes `submission_baseline.csv`, `submission_deep.csv` and `submission.csv`
(= the model chosen in `FINAL_CHOICE`).

**What it does, in order**
1. Cleans the training spectra (same filters as v0.1), canonicalises the molecules (the metric's recipe) and builds the **candidate library** = every distinct training molecule, with Morgan fingerprints and exact masses.
2. Builds spectrum features (fragment + neutral-loss blocks in mass-defect sub-bins, v0.2c recipe).
3. **Baseline:** Ridge regression (alpha 1) trained on the v0.1 subsample (≤5,000 molecules per source, ≤3 spectra each).
4. **Deep MLP:** a residual MLP trained on *all* molecules (≤3 spectra each).
5. For every test molecule: average the predicted fingerprints of its spectra, intersect their mass windows (instrument-offset-corrected, per adduct), keep library molecules in the window and rank them by cosine similarity to the predicted fingerprint.

**How to run on Kaggle**
- Settings → Accelerator: **GPU** (T4/P100) for the deep MLP (CPU works but is several times slower). Internet can stay off.
- RDKit must be 2026.03.3 (the metric's version). Use the notebook's package installer / dependency manager as in the baseline notebook if it is not preinstalled.
- First run with `DEBUG = True` (a few minutes) to check the whole pipeline; then set `DEBUG = False` for the real run (roughly 2–4 hours; progress and timings are printed).
- Everything expensive is cached in `/kaggle/working`, so a restarted session resumes quickly.

**Read this before interpreting the leaderboard score**
- The candidate library = every training molecule **plus ~432k COCONUT natural products** (a public database, pre-computed in the attached dataset; `EXTRA_POOL_DIR`). The visible `test.parquet` is a placeholder drawn from the training spectra (all its spectra have exact twins in train), so a score on it says nothing; only the hidden-test rerun counts. A molecule that is in neither the training set nor COCONUT cannot be found.
- Local benchmark: enlarging the pool with COCONUT costs about 0.11 MRR on natural-product-like validation molecules (more same-mass isomers), because COCONUT-only molecules have no spectra and cannot be validated locally.
- Validation results from the project (see the repo's EXPERIMENT_LOG) were measured with an oracle library that contains the true molecule; the sanity check at the end of each model section does the same on a held-out slice of the training data.

In [ ]:
# ======================================== 1. CONFIG ========================================
import os, sys, glob, time, json, gc, queue, threading, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
T0 = time.time()
def mem_limit_gb():
    """RAM this notebook may really use: the container (cgroup) limit if there is one, else the machine's RAM. psutil alone reports the HOST."""
    import psutil
    lim = psutil.virtual_memory().total
    for f in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            v = open(f).read().strip()
            if v.isdigit() and int(v) < lim: lim = int(v)
        except OSError: pass
    return lim / 1e9
def rss_gb():
    try:
        import psutil; return psutil.Process().memory_info().rss / 1e9
    except Exception: return float("nan")
def release_memory():
    """Hand freed heap back to the OS (glibc keeps it otherwise, so 'del big_table' does not lower the process size)."""
    gc.collect()
    try:
        import ctypes; ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass
def tick(msg): print(f"[{(time.time() - T0) / 60:6.1f} min | {rss_gb():4.1f} GB] {msg}", flush=True)

DEBUG = False                # <-- True: tiny end-to-end run (a few minutes) to check the pipeline. False: the real run (hours).
if os.environ.get("CASMI_DEBUG"): DEBUG = bool(int(os.environ["CASMI_DEBUG"]))      # (override used for local testing)
RUN_BASELINE = True          # Ridge baseline
RUN_DEEP = True              # deep residual MLP
FINAL_CHOICE = "deep"        # which model is copied to submission.csv: "baseline" or "deep"
EXTRA_CANDIDATES_PATH = None # optional .parquet/.csv with a 'smiles' column of extra candidate structures. (canonicalised in the notebook: slow for >100k rows; prefer EXTRA_POOL_DIR).
EXTRA_POOL_DIR = "auto"      # folder with the PRE-COMPUTED COCONUT cache (coconut_v1.parquet + coconut_fp_v1.npy from src/build_coconut_cache.py).
                             # "auto": look in /kaggle/input/** then ./artifacts. None: train-only library. Allowed: the Code Requirements permit freely and publicly available external data.
POP_WEIGHT = 0.0             # P7c popularity prior: score = z(cosine within the mass window) + POP_WEIGHT * pop. 0.0 = off (pure cosine, = P7b). Public pipelines use ~0.15.
POP_PATH = "auto"            # pool_popularity.csv of the public Kaggle dataset dmitriigluzdov/casmi26-pubchem-popularity-prior ("auto": search /kaggle/input/** then ./data/external/pop_prior).
FREEZE = True                # after training, save the model + library + feature settings into <WORK>/frozen: input of the inference-only notebook (casmi_infer)
RESTORE_CACHE_DIR = None     # None | "auto" | folder: copy canon_cache.parquet / X_train.npz / deep_pred.npz / deep_best.pt from an earlier run's output (attach that notebook
                             # output as an input) so this run skips canonicalisation, feature building and training. Only valid if the training data and code are unchanged.
if os.environ.get("CASMI_POP_WEIGHT"): POP_WEIGHT = float(os.environ["CASMI_POP_WEIGHT"])      # (override used for local testing)
SEED = 42

# --- locate the data (Kaggle input folder, or ./data when run locally) ---
def find_comp_dir():
    if os.environ.get("CASMI_COMP_DIR"): return os.environ["CASMI_COMP_DIR"]
    hits = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
    return os.path.dirname(hits[0]) if hits else "./data"
COMP_DIR = find_comp_dir()
TRAIN_PATH, TEST_PATH, SAMPLE_PATH = (os.path.join(COMP_DIR, f) for f in ("train.parquet", "test.parquet", "sample_submission.csv"))
WORK = os.environ.get("CASMI_WORK") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else "./kaggle_work")
os.makedirs(WORK, exist_ok=True)

# --- sizes (DEBUG shrinks everything) ---
N_DEBUG_MOLECULES = 6000
BASE_CAP_PER_SOURCE = 400 if DEBUG else 5000        # baseline = v0.1 recipe: <= 5,000 molecules per source
HOLDOUT_FRACTION = 0.10 if DEBUG else 0.03          # scaffold-grouped slice kept out of BOTH models for the sanity check
DEEP_MAX_EPOCHS, DEEP_PATIENCE = (2, 2) if DEBUG else (16, 4)

import rdkit
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")
print("Python", sys.version.split()[0], "| RDKit", rdkit.__version__, "(the metric pins 2026.03.3)")
if not str(rdkit.__version__).startswith("2026.03"): print("WARNING: RDKit version differs from the metric's; canonical keys may differ slightly.")
import torch
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, "| DEBUG =", DEBUG, "| data:", COMP_DIR, "| work:", WORK)
assert os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH), f"train/test parquet not found under {COMP_DIR}"
try:
    import psutil; print(f"CPU cores: {os.cpu_count()} | RAM: {psutil.virtual_memory().total / 1e9:.0f} GB on the host, usable by this notebook (container limit): {mem_limit_gb():.0f} GB")
except ImportError:
    print("CPU cores:", os.cpu_count())
if DEVICE == "cpu" and not DEBUG: print("WARNING: no GPU found - the deep MLP will be several times slower (enable Settings > Accelerator > GPU).")


In [ ]:
# ======================================== 1a. OPTIONAL: RESTORE CACHES FROM AN EARLIER RUN ========================================
# WHY: canonicalisation (~22 min on Kaggle), feature building and the deep MLP training (~1 h) do not depend on the library or the
# popularity prior. Attaching a previous notebook's output and setting RESTORE_CACHE_DIR = "auto" lets a rerun skip all of them.
if RESTORE_CACHE_DIR:
    import shutil
    d_ = RESTORE_CACHE_DIR
    if d_ == "auto":
        hits = glob.glob("/kaggle/input/**/deep_best.pt", recursive=True)
        d_ = os.path.dirname(hits[0]) if hits else None
    if d_ is None: print("WARNING: RESTORE_CACHE_DIR = 'auto' but no earlier output (deep_best.pt) is attached -> everything is recomputed")
    else:
        for f_ in ("canon_cache.parquet", "X_train.npz", "deep_pred.npz", "deep_best.pt"):
            src_, dst_ = os.path.join(d_, f_), os.path.join(WORK, f_)
            if os.path.exists(src_) and not os.path.exists(dst_): shutil.copy(src_, dst_); tick(f"restored {f_} from {d_}")


In [ ]:
# ======================================== 1b. PRE-FLIGHT: can this machine build the Ridge Gram matrix? ========================================
# WHY: on Kaggle the kernel died ~2 s into the Ridge fit (after 28 min of setup) although the process used only 3 of 32 GB. Whatever
# kills it, find out NOW (seconds, not 28 minutes): each size is tried in a throw-away subprocess, so a crash there cannot kill this
# notebook. The largest column count that survives becomes the cap for the Ridge fit (the model is unchanged if the full size works).
import subprocess
RIDGE_MAX_COLS = 10 ** 9
_PROBE = (
    "import sys, numpy as np, scipy.linalg.blas as B\n"
    "p = int(sys.argv[1]); rng = np.random.default_rng(0)\n"
    "G = np.zeros((p, p), order='F')\n"                                   # same allocation as the real fit
    "A = rng.random((200, p))\n"
    "G = B.dsyrk(1.0, A, trans=1, beta=1.0, c=G, lower=1, overwrite_c=1)\n"
    "G[::7] += 0.0\n"                                                     # touch memory like the centring step does
    "G[np.diag_indices_from(G)] += 1.0\n"
    "print('probe ok', p, G.nbytes / 1e9, 'GB')\n")
if RUN_BASELINE:
    _ladder = [3000] if DEBUG else [30000, 24000, 18000, 12000, 8000]
    for _p in _ladder:
        _r = subprocess.run([sys.executable, "-c", _PROBE, str(_p)], capture_output=True, text=True, timeout=900)
        if _r.returncode == 0:
            RIDGE_MAX_COLS = _p; print(f"Ridge pre-flight OK at {_p:,} columns ({_p * _p * 8 / 1e9:.1f} GB Gram matrix)"); break
        print(f"Ridge pre-flight FAILED at {_p:,} columns: return code {_r.returncode} (-9 = killed for memory, -11 = crash). stderr: {_r.stderr[-300:]!r}")
    else:
        raise RuntimeError("the Ridge Gram matrix cannot be built even at 8,000 columns on this machine; run only the deep notebook")


In [ ]:
# ======================================== 2. MASS HELPERS ========================================
# ion m/z = neutral mass + shift. [M+K]+ and [M+Cl]- appear in the test file (4 spectra) but not in the v0.1 cleaning table.
SHIFT = {"[M+H]+": 1.007276, "[M-H]-": -1.007276, "[M+Na]+": 22.989218, "[M+NH4]+": 18.033823,
         "[M+CH2O2-H]-": 44.997655, "[M+K]+": 38.963158, "[M+Cl]-": 34.969402}
V01_ADDUCTS = ["[M+H]+", "[M-H]-", "[M+Na]+", "[M+NH4]+", "[M+CH2O2-H]-"]     # cleaning table (dimers excluded)

import re
_FORMULA, _fm_cache = re.compile(r"([A-Z][a-z]?)(\d*)"), {}
def formula_mass(formula):
    """Monoisotopic mass of a neutral formula string such as 'C20H28N2O3' (NaN if unparsable)."""
    if formula in _fm_cache: return _fm_cache[formula]
    pt = Chem.GetPeriodicTable()
    try:
        if not re.fullmatch(r"([A-Z][a-z]?\d*)+", formula): raise ValueError
        m = sum(pt.GetMostCommonIsotopeMass(el) * (int(n) if n else 1) for el, n in _FORMULA.findall(formula))
    except Exception:
        m = np.nan
    _fm_cache[formula] = m
    return m

def fit_bias(err_ppm, adduct):
    """Per-adduct calibration offset (median signed error, ppm) and robust spread (1.4826 * MAD)."""
    g = pd.DataFrame({"err": err_ppm, "adduct": adduct}).groupby("adduct").err
    return pd.DataFrame({"n": g.size(), "bias_ppm": g.median(), "sd_ppm": g.apply(lambda s: 1.4826 * np.median(np.abs(s - s.median())))})

def halfwidth_by_adduct(tab, k=3.5, floor=3.0):
    """Per-adduct half-width in ppm: k robust sigmas, never below `floor`."""
    return (k * tab.sd_ppm).clip(lower=floor).to_dict()

def mass_window(nm, adduct, tol_ppm, bias):
    """Candidate-mass interval [lo, hi] per query: window centred on the offset-corrected neutral mass nm / (1 + offset)."""
    nm = np.asarray(nm, float)
    nm = nm / (1 + pd.Series(adduct).map(bias).fillna(0.0).values * 1e-6)
    tol = pd.Series(adduct).map(tol_ppm).fillna(max(tol_ppm.values())).values
    return nm * (1 - tol * 1e-6), nm * (1 + tol * 1e-6)

class Library:
    """Candidate pool: unit-length fingerprints for cosine ranking and a mass-sorted index for the window search."""
    def __init__(self, smiles, keys, mass, fp_unit):
        self.smiles, self.keys, self.mass, self.fp_unit = smiles, np.asarray(keys), np.asarray(mass), fp_unit
        self.order = np.argsort(self.mass); self.mass_sorted = self.mass[self.order]
        self.pop, self.pop_w = None, 0.0        # optional popularity prior (one value per library row) and its weight; None = pure cosine

def unit(A): return A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)


In [ ]:
# ======================================== 3. TRAINING DATA: clean, select ========================================
# WHY: only the small columns are read here (not the peak arrays); peaks are streamed later, one row group at a time.
import pyarrow.parquet as pq
cols = ["ingest_lib", "normalized_smiles", "inchikey14", "molecular_formula", "adduct", "precursor_mz", "precursor_error_ppm", "num_peaks"]
raw = pq.read_table(TRAIN_PATH, columns=cols).to_pandas()            # row position in the file = row id (used to stream peaks)
tick(f"train metadata: {len(raw):,} spectra")

# --- instrument calibration offsets (P3 finding): precursor_error_ppm is unsigned, so recompute the SIGNED error from the formula.
#     enveda-180 is the same instrument/processing as the test file. Offsets: [M+H]+ ~ +2.2 ppm, [M-H]- ~ +0.8 ppm. ---
e180 = raw[(raw.ingest_lib == "enveda-180") & raw.adduct.isin(SHIFT)].copy()
if len(e180) == 0: e180 = raw[raw.adduct.isin(SHIFT)].copy()      # (only if enveda-180 is absent, e.g. a tiny debug subset)
e180["M"] = e180.molecular_formula.map(formula_mass); e180 = e180.dropna(subset=["M"])
e180["err"] = ((e180.precursor_mz - e180.adduct.map(SHIFT)) - e180.M) / e180.M * 1e6
BIAS_TAB = fit_bias(e180.err.values, e180.adduct.values)
BIAS, HW = BIAS_TAB.bias_ppm.to_dict(), halfwidth_by_adduct(BIAS_TAB)
print(BIAS_TAB.round(2)); del e180

# --- v0.1 cleaning filters ---
df = raw[raw.adduct.isin(V01_ADDUCTS) & raw.precursor_mz.between(50, 1500) & raw.precursor_error_ppm.abs().le(20)
         & raw.num_peaks.between(3, 500)].rename_axis("row").reset_index()
del raw; gc.collect()
tick(f"cleaned: {len(df):,} spectra, {df.inchikey14.nunique():,} molecules")

rng = np.random.default_rng(SEED)
if DEBUG:
    pick = rng.choice(df.inchikey14.unique(), N_DEBUG_MOLECULES, replace=False)
    df = df[df.inchikey14.isin(pick)].reset_index(drop=True)
    tick(f"DEBUG subset: {len(df):,} spectra, {df.inchikey14.nunique():,} molecules")

# --- deep-model set = every molecule, <= 3 spectra each (random spectra, seeded) ---
deep = df.sample(frac=1, random_state=0).groupby("inchikey14").head(3).reset_index(drop=True)
# --- baseline set = the v0.1 recipe: <= BASE_CAP_PER_SOURCE molecules per source ---
mol_src = deep.groupby("inchikey14").ingest_lib.first()
base_prov = set()
for s_name, g in mol_src.groupby(mol_src):
    base_prov |= set(rng.choice(g.index.values, min(BASE_CAP_PER_SOURCE, len(g)), replace=False))
deep["in_base"] = deep.inchikey14.isin(base_prov)
del df; gc.collect()
tick(f"deep set {len(deep):,} spectra / {deep.inchikey14.nunique():,} molecules | baseline set {int(deep.in_base.sum()):,} spectra")


In [ ]:
# ======================================== 4. CANONICALISE (the metric's recipe), cached ========================================
# WHY: the answer is scored by InChIKey14 of the RDKit canonical TAUTOMER, so keys/scaffolds must be computed the same way.
# Scaffold (stereo removed) is only used to keep the sanity holdout scaffold-disjoint from training.
from multiprocessing import get_context
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
_taut = None
def key_and_scaffold(smi):
    global _taut
    try:
        if _taut is None: _taut = rdMolStandardize.TautomerEnumerator()
        m = Chem.MolFromSmiles(smi)
        if m is None: return None, None
        mc = _taut.Canonicalize(m)
        key = Chem.MolToInchiKey(mc)[:14]
        ms = Chem.Mol(mc); Chem.RemoveStereochemistry(ms)
        return key, MurckoScaffold.MurckoScaffoldSmiles(mol=ms)
    except Exception:
        return None, None

def canonicalise(smiles_list, cache_path):
    cache = pd.read_parquet(cache_path) if os.path.exists(cache_path) else pd.DataFrame(columns=["smiles", "key", "scaffold"])
    have = set(cache.smiles); todo = [s for s in dict.fromkeys(smiles_list) if s not in have]
    tick(f"canonicalising {len(todo):,} SMILES ({len(cache):,} cached) on {os.cpu_count()} cores")
    if todo:
        with get_context("fork").Pool(os.cpu_count()) as pool:
            res = pool.map(key_and_scaffold, todo, chunksize=200)
        cache = pd.concat([cache, pd.DataFrame({"smiles": todo, "key": [r[0] for r in res], "scaffold": [r[1] for r in res]})])
        cache.to_parquet(cache_path)
    return cache

cache = canonicalise(deep.normalized_smiles.tolist(), f"{WORK}/canon_cache{'_debug' if DEBUG else ''}.parquet")
deep = deep.merge(cache, left_on="normalized_smiles", right_on="smiles", how="left").dropna(subset=["key", "scaffold"]).reset_index(drop=True)
deep["scaffold"] = deep.groupby("key").scaffold.transform(lambda s: s.mode().iloc[0])
empty = deep.scaffold == ""
deep.loc[empty, "scaffold"] = "acyclic_" + deep.loc[empty, "key"]          # every acyclic molecule is its own group
deep = deep.groupby("key").head(3).reset_index(drop=True)                    # <= 3 spectra per CANONICAL molecule
# a canonical molecule is in the baseline set if any of its provided keys was sampled
base_keys = set(deep[deep.in_base].key); deep["in_base"] = deep.key.isin(base_keys)

# --- sanity holdout: scaffold-grouped, excluded from BOTH models (so the sanity score is an honest generalisation check) ---
from sklearn.model_selection import GroupShuffleSplit
tr_i, ho_i = next(GroupShuffleSplit(n_splits=1, test_size=HOLDOUT_FRACTION, random_state=SEED).split(deep, groups=deep.scaffold))
deep["split"] = "train"; deep.loc[ho_i, "split"] = "holdout"
tick(f"canonical molecules: {deep.key.nunique():,} | train spectra {int((deep.split == 'train').sum()):,} | holdout spectra {int((deep.split == 'holdout').sum()):,} "
     f"({deep[deep.split == 'holdout'].key.nunique():,} molecules)")


In [ ]:
# ======================================== 4b. EXTERNAL POOL + POPULARITY PRIOR (shared by the training and the inference notebook) ========================================
def merge_coconut(lib_smiles, lib_keys, fp_u8, mass):
    """Append the pre-computed COCONUT cache (EXTRA_POOL_DIR) to the library arrays. Structures already in the library win; one row per canonical key."""
    d_ = EXTRA_POOL_DIR
    if not d_: return lib_smiles, lib_keys, fp_u8, mass
    if d_ == "auto":
        hits = glob.glob("/kaggle/input/**/coconut_v1.parquet", recursive=True) + glob.glob("artifacts/coconut_v1.parquet")
        d_ = os.path.dirname(hits[0]) if hits else None
    if d_ is None:
        print("WARNING: no COCONUT cache found -> library = training molecules only (expect a score near 0 on novel molecules)")
        return lib_smiles, lib_keys, fp_u8, mass
    C_ = pd.read_parquet(f"{d_}/coconut_v1.parquet"); cfp = np.load(f"{d_}/coconut_fp_v1.npy", mmap_mode="r")
    assert len(C_) == len(cfp) and cfp.shape[1] == 1024, "COCONUT cache rows do not match its fingerprints"
    keep = ~C_.key.isin(set(lib_keys)).values & ~C_.key.duplicated().values        # training structures win; one row per canonical key
    idx_ = np.flatnonzero(keep); C_ = C_.iloc[idx_]
    out = (list(lib_smiles) + C_.smiles.tolist(), list(lib_keys) + C_.key.tolist(), np.vstack([fp_u8, np.asarray(cfp[idx_])]), np.concatenate([mass, C_.mass.values]))
    tick(f"+ {len(C_):,} COCONUT candidates from {d_}")
    return out

def attach_popularity(LIB, weight):
    """P7c: PubChem documentation prior, ln(1+substances)+ln(1+PubMed)+..., joined on the metric key; unmatched keys get the median (neutral)."""
    hits = glob.glob("/kaggle/input/**/pool_popularity.csv", recursive=True) + glob.glob("data/external/pop_prior/pool_popularity.csv")
    path = hits[0] if POP_PATH == "auto" and hits else (POP_PATH if POP_PATH != "auto" else None)
    if path is None: raise FileNotFoundError("POP_WEIGHT > 0 but pool_popularity.csv was not found: attach the dataset dmitriigluzdov/casmi26-pubchem-popularity-prior")
    pp = pd.read_csv(path, usecols=["inchikey14", "pop"]).drop_duplicates("inchikey14").set_index("inchikey14")["pop"]
    pop_ = pp.reindex(LIB.keys).values.astype(np.float32); miss_ = np.isnan(pop_)
    pop_[miss_] = np.nanmedian(pop_)
    LIB.pop, LIB.pop_w = pop_, float(weight)
    tick(f"popularity prior ON: weight {weight}, {1 - miss_.mean():.1%} of the library matched, pop median {np.median(pop_):.2f}")


In [ ]:
# ======================================== 5. CANDIDATE LIBRARY ========================================
# WHY: the model predicts a 1,024-bit Morgan fingerprint; candidates are library molecules whose exact mass fits the precursor and
# whose fingerprint is closest (cosine) to the prediction. Library = every distinct training molecule (+ optional extra structures).
from rdkit.Chem import Descriptors, rdFingerprintGenerator
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def fp_and_mass(smiles):
    fp = np.zeros((len(smiles), 1024), np.uint8); mass = np.full(len(smiles), np.nan); ok = np.zeros(len(smiles), bool)
    for i, s in enumerate(smiles):
        m = Chem.MolFromSmiles(s)
        if m is None: continue
        fp[i] = fpgen.GetFingerprintAsNumPy(m); mass[i] = Descriptors.ExactMolWt(m); ok[i] = True
    return fp, mass, ok

mols = deep.drop_duplicates("key")[["key", "normalized_smiles"]].reset_index(drop=True)
fp_u8, mass, ok = fp_and_mass(mols.normalized_smiles.tolist())
mols, fp_u8, mass = mols[ok].reset_index(drop=True), fp_u8[ok], mass[ok]
MOL_IDX = pd.Series(np.arange(len(mols)), index=mols.key)            # canonical key -> row in fp_u8 (training targets)
deep = deep[deep.key.isin(MOL_IDX.index)].reset_index(drop=True)      # (RDKit parse failures, if any, cannot be training targets)
tick(f"library from training molecules: {len(mols):,}")

lib_smiles, lib_keys = mols.normalized_smiles.tolist(), mols.key.tolist()
N_TRAIN_LIB = len(lib_keys)                                            # rows [:N_TRAIN_LIB] of the library are the training molecules (saved by the freeze step)
if EXTRA_CANDIDATES_PATH:                                              # optional external structures (check the competition rules first!)
    ex = pd.read_parquet(EXTRA_CANDIDATES_PATH) if EXTRA_CANDIDATES_PATH.endswith(".parquet") else pd.read_csv(EXTRA_CANDIDATES_PATH)
    ec = canonicalise(ex.smiles.tolist(), f"{WORK}/canon_extra_candidates.parquet")
    ex = ex.merge(ec, left_on="smiles", right_on="smiles", how="left").dropna(subset=["key"]).drop_duplicates("key")
    ex = ex[~ex.key.isin(set(lib_keys))]
    efp, emass, eok = fp_and_mass(ex.smiles.tolist()); ex, efp, emass = ex[eok], efp[eok], emass[eok]
    lib_smiles += ex.smiles.tolist(); lib_keys += ex.key.tolist(); fp_u8 = np.vstack([fp_u8, efp]); mass = np.concatenate([mass, emass])
    tick(f"+ {len(ex):,} extra candidates")

lib_smiles, lib_keys, fp_u8, mass = merge_coconut(lib_smiles, lib_keys, fp_u8, mass)        # pre-computed external pool (COCONUT, ~432k natural products, no spectra)
LIB = Library(lib_smiles, lib_keys, mass, unit(fp_u8.astype(np.float32)))
if POP_WEIGHT > 0: attach_popularity(LIB, POP_WEIGHT)
tick(f"library ready: {len(LIB.keys):,} molecules, mass {LIB.mass_sorted[0]:.1f}-{LIB.mass_sorted[-1]:.1f}")


In [ ]:
# ======================================== 6. SPECTRUM FEATURES ========================================
# WHY (project findings P4a-c): features = fragment peaks + NEUTRAL LOSSES (precursor m/z - fragment m/z), each binned as
# nominal mass x 0.02 Da MASS-DEFECT sub-bins (the digits after the decimal point carry elemental composition), sqrt intensity,
# unit length per block, plus acquisition metadata. Sparse matrices (about 130 non-zeros per spectrum).
import scipy.sparse as sp
import pyarrow.parquet as pq
MAX_MZ, DEFECT_LO, DEFECT_HI, W = 1000, -0.05, 0.25, 0.02
K_SUB = int(round((DEFECT_HI - DEFECT_LO) / W)); PER = K_SUB + 1          # 15 sub-bins + 1 overflow column per nominal mass

def defect_block(axes, its, nb_nominal=MAX_MZ, eps=1e-6):
    """Nominal n = floor(x + 0.05); defect d = x - n; d in [-0.05, 0.25) -> one of 15 sub-bins, else an overflow column."""
    r_, c_, v_ = [], [], []
    for r, (a, i) in enumerate(zip(axes, its)):
        a = np.asarray(a, float); n = np.floor(a + 0.05 + eps).astype(int); d = a - n
        ok = (a >= 0) & (n < nb_nominal)
        if not ok.any(): continue
        sub = np.where((d >= DEFECT_LO) & (d < DEFECT_HI), np.floor((d - DEFECT_LO) / W + eps).astype(int).clip(0, K_SUB - 1), K_SUB)
        u, inv = np.unique((n * PER + sub)[ok], return_inverse=True)
        v = np.bincount(inv, weights=np.sqrt(np.asarray(i, float)[ok])); v /= np.linalg.norm(v) + 1e-9
        r_.append(np.full(len(u), r)); c_.append(u); v_.append(v)
    if not r_: return sp.csr_matrix((len(axes), nb_nominal * PER), dtype=np.float32)
    return sp.csr_matrix((np.concatenate(v_).astype(np.float32), (np.concatenate(r_), np.concatenate(c_))), shape=(len(axes), nb_nominal * PER))

def ce_scalar(x):
    """Collision energy: list -> mean and flag 1 (every row stores a list; the flag therefore means 'has a CE value')."""
    if x is None: return np.nan, 0.0
    a = np.asarray(x, float); return (np.nan if a.size == 0 else float(np.nanmean(a))), 1.0

class Features:
    def __init__(self, fit_df):
        self.top_inst = fit_df.instrument_type.value_counts().head(8).index
        self.cols = self._meta(fit_df).columns; self.ce_med = fit_df.collision_energy_ev.median()
    def _meta(self, d):
        m = d[["ionization_mode", "instrument_type", "adduct"]].copy()
        m["instrument_type"] = m.instrument_type.where(m.instrument_type.isin(self.top_inst), "other")
        return pd.get_dummies(m)
    def build(self, d, mzs, its):
        frag = defect_block(mzs, its)
        loss = defect_block([np.where(p - np.asarray(m) >= 1.0, p - np.asarray(m), -1.0) for m, p in zip(mzs, d.precursor_mz.values)], its)
        M = self._meta(d).reindex(columns=self.cols, fill_value=0).values.astype(np.float32)
        num = np.c_[d.neutral_mass / MAX_MZ, d.collision_energy_ev.fillna(self.ce_med) / 100, d.ce_multi].astype(np.float32)
        return sp.hstack([frag, loss, sp.csr_matrix(np.hstack([num, M]))], format="csr")

def with_meta(frame, t):
    """Attach the per-spectrum acquisition columns read from the parquet (t = pandas frame with the same length/order)."""
    d = frame[["adduct", "precursor_mz"]].reset_index(drop=True).copy()
    d["instrument_type"], d["ionization_mode"] = t.instrument_type.values, t.ionization_mode.values
    ce = [ce_scalar(x) for x in t.collision_energy_ev]
    d["collision_energy_ev"], d["ce_multi"] = [c[0] for c in ce], np.float32([c[1] for c in ce])
    d["neutral_mass"] = d.precursor_mz - d.adduct.map(SHIFT)               # n_molecules = 1 for every monomer adduct
    return d

def read_rows_meta(row_ids):
    """instrument_type / ionization_mode / collision energy for selected rows (small columns only)."""
    pf = pq.ParquetFile(TRAIN_PATH); starts = np.cumsum([0] + [pf.metadata.row_group(i).num_rows for i in range(pf.num_row_groups)])
    row_ids = np.asarray(row_ids); parts = []
    for g in range(pf.num_row_groups):
        m = np.flatnonzero((row_ids >= starts[g]) & (row_ids < starts[g + 1]))
        if len(m):
            t = pf.read_row_group(g, columns=["instrument_type", "ionization_mode", "collision_energy_ev"]).take(row_ids[m] - starts[g]).to_pandas()
            t.index = m; parts.append(t)
    return pd.concat(parts).sort_index().reset_index(drop=True)


In [ ]:
# ======================================== 6b. BUILD THE TRAINING FEATURES ========================================
# the Ridge-only notebook never uses most spectra: keep only its training rows + the sanity holdout (saves minutes of feature building).
# (The candidate library above was already built from ALL molecules.)
if not RUN_DEEP:
    deep = deep[deep.in_base | (deep.split == "holdout")].reset_index(drop=True)
    tick(f"baseline-only run: features for {len(deep):,} spectra instead of all of them")

# fit the feature scaler/one-hot columns on the TRAINING rows only
tr_meta = with_meta(deep[deep.split == "train"], read_rows_meta(deep[deep.split == "train"].row.values))
FEAT = Features(tr_meta); del tr_meta
tick(f"feature object fitted ({len(FEAT.cols)} metadata columns, CE median {FEAT.ce_med:.1f})")

def stream_features(frame):
    """One pass over the parquet row groups: peaks -> sparse features, never holding all peaks in memory."""
    pf = pq.ParquetFile(TRAIN_PATH); starts = np.cumsum([0] + [pf.metadata.row_group(i).num_rows for i in range(pf.num_row_groups)])
    rid = frame.row.values; parts, pos = [], []
    for g in range(pf.num_row_groups):
        m = np.flatnonzero((rid >= starts[g]) & (rid < starts[g + 1]))
        if len(m) == 0: continue
        t = (pf.read_row_group(g, columns=["ms2_mzs", "ms2_normalized_intensities", "collision_energy_ev", "instrument_type", "ionization_mode"])
             .take(rid[m] - starts[g]).to_pandas())
        d = with_meta(frame.iloc[m], t)
        parts.append(FEAT.build(d, t.ms2_mzs.values, t.ms2_normalized_intensities.values).tocsr()); pos.append(m)
    return sp.vstack(parts).tocsr()[np.argsort(np.concatenate(pos))]

xpath = f"{WORK}/X_train{'_debug' if DEBUG else ''}.npz"
if os.path.exists(xpath) and sp.load_npz(xpath).shape[0] == len(deep):
    X_ALL = sp.load_npz(xpath); tick(f"features loaded from cache {X_ALL.shape}")
else:
    X_ALL = stream_features(deep); sp.save_npz(xpath, X_ALL); tick(f"features built {X_ALL.shape}, {X_ALL.nnz / X_ALL.shape[0]:.0f} non-zeros per spectrum")
Y_ALL = fp_u8[MOL_IDX[deep.key].values]                                # uint8 fingerprint target, one row per spectrum
release_memory(); tick("memory released after feature building")


In [ ]:
# ======================================== 7. TEST SPECTRA ========================================
test = pd.read_parquet(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
assert set(sample.molecule_id) == set(test.molecule_id), "sample_submission ids differ from the test file"
td = with_meta(test, test)                     # test has the same columns (adduct, precursor_mz, instrument_type, ionization_mode, collision_energy_ev)
X_TEST = FEAT.build(td, test.ms2_mzs.values, test.ms2_normalized_intensities.values).tocsr()
test_lo, test_hi = mass_window(td.neutral_mass.values, test.adduct.values, HW, BIAS)
tick(f"test: {len(test):,} spectra, {test.molecule_id.nunique():,} molecules; adducts {test.adduct.value_counts().to_dict()}")
print("spectra with an adduct that has no calibration offset (window centred on zero):", int((~test.adduct.isin(BIAS)).sum()))


In [ ]:
# ======================================== 8. RANKING + SUBMISSION WRITER ========================================
def rank_queries(Q, lo, hi, lib, k=25):
    """Q: unit-length predicted fingerprints (n x 1024); lo/hi: mass windows. Returns one array of library row indices per query,
    best first (<= k): library molecules whose exact mass lies in [lo, hi], ordered by cosine similarity to the query.
    Empty window -> widen around the window centre (10..250 ppm), finally fall back to the k nearest masses (a row must never be empty)."""
    out, ncand = [], np.zeros(len(Q), int)
    for i in range(len(Q)):
        a, b = np.searchsorted(lib.mass_sorted, [lo[i], hi[i]]); idx = lib.order[a:b]
        if len(idx) == 0:
            c = 0.5 * (lo[i] + hi[i])
            for ppm in (10, 25, 50, 100, 250):
                a, b = np.searchsorted(lib.mass_sorted, [c * (1 - ppm * 1e-6), c * (1 + ppm * 1e-6)])
                if b > a: idx = lib.order[a:b]; break
            if len(idx) == 0: idx = lib.order[np.argsort(np.abs(lib.mass_sorted - c))[:k]]
        ncand[i] = len(idx)
        s = lib.fp_unit[idx] @ Q[i]
        if lib.pop is not None and len(idx) > 1: s = (s - s.mean()) / (s.std() + 1e-6) + lib.pop_w * lib.pop[idx]      # P7c prior (off when lib.pop is None)
        if len(idx) > k:
            part = np.argpartition(-s, k)[:k]; order = part[np.argsort(-s[part])]
        else:
            order = np.argsort(-s)
        out.append(idx[order])
    return out, ncand

def pool_molecules(P, mol_ids, lo, hi):
    """One query per molecule_id: mean of the predicted fingerprints of its spectra (unit length) and the INTERSECTION of the spectra's
    mass windows (union if they do not overlap). Validated in the project: pooling beats scoring spectra one by one (+0.03 to +0.06 MRR)."""
    ids = pd.Series(mol_ids); groups = ids.groupby(ids).indices; keys = list(groups)
    Pm = unit(np.stack([P[groups[k]].mean(0) for k in keys]))
    L = np.array([lo[groups[k]].max() for k in keys]); H = np.array([hi[groups[k]].min() for k in keys])
    bad = L > H
    L[bad] = np.array([lo[groups[k]].min() for k in keys])[bad]; H[bad] = np.array([hi[groups[k]].max() for k in keys])[bad]
    return keys, Pm, L, H, int(bad.sum())

def validate_submission(sub, expected_ids):
    """The competition's rejection rules: columns present, non-empty, no nulls, no repeated molecule_id, <= 25 guesses per row."""
    assert list(sub.columns) == ["molecule_id", "smiles"], "need exactly the columns molecule_id,smiles"
    assert len(sub) > 0, "empty submission"
    assert sub.molecule_id.notna().all() and sub.smiles.notna().all(), "nulls present"
    assert (sub.smiles.astype(str).str.len() > 0).all(), "empty smiles field"
    assert sub.molecule_id.is_unique, "repeated molecule_id"
    assert set(sub.molecule_id) == set(expected_ids), "molecule_ids differ from the test set"
    n = sub.smiles.str.split(";").str.len()
    assert n.max() <= 25, f"more than 25 guesses in a row (max {n.max()})"
    return n

def publish(sub, label):
    """Write <WORK>/submission.csv: the file Kaggle picks up automatically when the notebook is submitted. Called as soon as the first model
    finishes (so a valid file always exists) and again when the final model is ready. Validated against the competition's rejection rules."""
    path = f"{WORK}/submission.csv"; sub.to_csv(path, index=False)
    validate_submission(pd.read_csv(path), sample.molecule_id)
    tick(f"submission.csv written from: {label}" + ("   (DEBUG run: tiny library, do NOT submit this one)" if DEBUG else ""))

def make_submission(P_test, name):
    """Predicted fingerprints per test SPECTRUM -> submission with one row per molecule_id."""
    keys, Pm, L, H, nbad = pool_molecules(P_test, test.molecule_id.values, test_lo, test_hi)
    ranked, ncand = rank_queries(Pm, L, H, LIB)
    sub = pd.DataFrame({"molecule_id": keys, "smiles": [";".join(LIB.smiles[j] for j in r) for r in ranked]})
    sub = sub.set_index("molecule_id").loc[sample.molecule_id].reset_index()            # same order as sample_submission
    n = validate_submission(sub, sample.molecule_id)
    path = f"{WORK}/submission_{name}{'_debug' if DEBUG else ''}.csv"; sub.to_csv(path, index=False)
    tick(f"{path}: {len(sub)} molecules | guesses per molecule: median {n.median():.0f}, min {n.min()} | candidates in window: median {np.median(ncand):.0f}, "
         f"empty windows {int((ncand == 0).sum())} | spectra windows that did not overlap: {nbad}")
    return sub, ranked, Pm

# holdout sanity check = the project's benchmark recipe: an ORACLE library (the true molecule is a candidate), per-spectrum queries,
# enveda-180 spectra only (the test instrument). Expect clearly non-zero scores; a score near 0 means a bug.
def holdout_sanity(P_hold, name, max_queries=3000):
    h = deep[deep.split == "holdout"].reset_index(drop=True)
    sel = np.flatnonzero((h.ingest_lib == "enveda-180").values)
    if len(sel) == 0: sel = np.arange(len(h))
    sel = sel[:max_queries]; hh = h.iloc[sel]
    nm = hh.precursor_mz.values - hh.adduct.map(SHIFT).values
    lo, hi = mass_window(nm, hh.adduct.values, HW, BIAS)
    ranked, ncand = rank_queries(unit(P_hold[sel]), lo, hi, LIB)
    rr = np.array([1 / (list(LIB.keys[r]).index(k) + 1) if k in set(LIB.keys[r]) else 0.0 for r, k in zip(ranked, hh.key.values)])
    tick(f"[{name}] holdout sanity (oracle library, {len(sel):,} enveda-180 spectra): MRR@25 = {rr.mean():.4f} | median candidates {np.median(ncand):.0f}")
    return rr.mean()


In [ ]:
# ======================================== 9. BASELINE MODEL: RIDGE ========================================
# Ridge(alpha=1) solved with the normal equations on sparse features (sklearn's sparse path fits one target at a time: far too slow
# for 1,024 targets). G = X'X is accumulated in row chunks (symmetric rank-k update, lower triangle) and factorised once (Cholesky).
import scipy.linalg
class NormalEqRidge:
    def __init__(self, alpha=1.0, chunk=2000): self.alpha, self.chunk = alpha, chunk
    def fit(self, X, Y):
        X = sp.csr_matrix(X); cnt = np.asarray((X != 0).sum(0)).ravel(); self.keep = np.flatnonzero(cnt > 0)     # drop columns empty in training
        # The Gram matrix (p x p float64, ~7 GB at the full 29.8k columns) must fit in the RAM that is left. If it would not, drop the rarest
        # columns (fewest training spectra) until it does. With enough RAM nothing is dropped and the model is exactly the benchmarked one.
        release_memory()
        room = (mem_limit_gb() - rss_gb() - 4.0) * 1e9 * 0.9          # 4 GB for chunks / X'Y / solve temporaries, 10% margin
        p_max = min(int(np.sqrt(max(room, 1e9) / 8)), RIDGE_MAX_COLS)       # RIDGE_MAX_COLS: what the pre-flight showed this machine can build
        if len(self.keep) > p_max:
            order = np.argsort(-cnt[self.keep], kind="stable"); kept = np.sort(self.keep[order[:p_max]])
            print(f"  Ridge: limited by free RAM / pre-flight (~{room / 1e9:.0f} GB free) -> keeping the {p_max:,} most frequent of {len(self.keep):,} columns (rarest dropped; occur in <= {cnt[kept].min()} spectra)", flush=True)
            self.keep = kept
        X = X[:, self.keep].astype(np.float64); n, p = X.shape
        print(f"  Ridge: {n:,} rows x {p:,} columns, Gram matrix {p * p * 8 / 1e9:.1f} GB (process now {rss_gb():.1f} GB of {mem_limit_gb():.0f} GB)", flush=True)
        G = np.zeros((p, p), order="F")
        for i in range(0, n, self.chunk):
            G = scipy.linalg.blas.dsyrk(1.0, X[i:i + self.chunk].toarray(), trans=1, beta=1.0, c=G, lower=1, overwrite_c=1)
        self.mu = np.asarray(X.mean(0)).ravel()
        for i in range(0, p, 2000): G[i:i + 2000] -= n * np.outer(self.mu[i:i + 2000], self.mu)                  # centre (intercept)
        XtY, ysum = np.zeros((p, Y.shape[1])), np.zeros(Y.shape[1])
        for i in range(0, n, 20000):
            Yc = np.asarray(Y[i:i + 20000], np.float64); XtY += np.asarray(X[i:i + 20000].T @ Yc); ysum += Yc.sum(0)
        self.ymu = ysum / n; XtY -= n * np.outer(self.mu, self.ymu)
        G[np.diag_indices_from(G)] += self.alpha
        c = scipy.linalg.cho_factor(G, lower=True, overwrite_a=True, check_finite=False)
        self.W = scipy.linalg.cho_solve(c, XtY, check_finite=False).astype(np.float32); del G, c
        self.b = (self.ymu - self.mu @ self.W).astype(np.float32)
        return self
    def predict(self, X): return np.asarray((sp.csr_matrix(X)[:, self.keep] @ self.W) + self.b, np.float32)

SPLIT_TRAIN = (deep.split == "train").values; SPLIT_HOLD = (deep.split == "holdout").values
BASELINE_FILE = f"{WORK}/ridge_pred{'_debug' if DEBUG else ''}.npz"
if RUN_BASELINE:
    if os.path.exists(BASELINE_FILE):
        z = np.load(BASELINE_FILE); P_test_ridge, P_hold_ridge = z["test"], z["hold"]; tick("baseline predictions loaded from cache")
    else:
        m = SPLIT_TRAIN & deep.in_base.values
        tick(f"fitting Ridge on {int(m.sum()):,} spectra ({deep[m].key.nunique():,} molecules)")
        ridge = NormalEqRidge(alpha=1.0).fit(X_ALL[m], Y_ALL[m])
        P_test_ridge, P_hold_ridge = ridge.predict(X_TEST), ridge.predict(X_ALL[SPLIT_HOLD])
        np.savez(BASELINE_FILE, test=P_test_ridge, hold=P_hold_ridge); del ridge; gc.collect()
        tick("Ridge fitted")
    holdout_sanity(P_hold_ridge, "baseline Ridge")
    SUB_BASELINE, RANKED_BASELINE, _ = make_submission(P_test_ridge, "baseline")
    publish(SUB_BASELINE, "baseline Ridge (provisional: replaced by the deep model below if FINAL_CHOICE = 'deep')")


In [ ]:
# ======================================== 10. DEEP MODEL: RESIDUAL MLP ========================================
# sparse spectrum features -> Linear(1024) -> 4 pre-norm residual blocks (LayerNorm) -> 1,024 sigmoid outputs, trained with BCE on the
# Morgan bits. Early stopping on a scaffold-grouped 5% holdout INSIDE the training rows (the sanity holdout is never touched).
# Project finding: with ~10x more data than the first experiments this model overtakes Ridge by a wide margin.
import torch.nn as nn
class ResBlock(nn.Module):
    def __init__(self, d, dropout):
        super().__init__(); self.norm = nn.LayerNorm(d)
        self.f = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d, d))
    def forward(self, x): return x + self.f(self.norm(x))

def make_net(n_in, d=1024, blocks=4, dropout=0.3, n_out=1024):
    return nn.Sequential(nn.Linear(n_in, d), nn.ReLU(), nn.Dropout(dropout), *[ResBlock(d, dropout) for _ in range(blocks)], nn.LayerNorm(d), nn.Linear(d, n_out))

def batches(X, Y, index_batches, device):
    """Background thread densifies the sparse rows while the accelerator works on the previous batch."""
    q = queue.Queue(maxsize=6)
    def work():
        for b in index_batches:
            xb = torch.from_numpy(X[b].toarray()); yb = torch.from_numpy(Y[b]) if Y is not None else None
            if device == "cuda": xb = xb.pin_memory()
            q.put((xb, yb))
        q.put(None)
    threading.Thread(target=work, daemon=True).start()
    while True:
        item = q.get()
        if item is None: return
        yield item[0].to(device), (item[1].to(device) if item[1] is not None else None)

def fit_mlp(X, Y, scaffolds, max_epochs, patience, lr=1e-3, weight_decay=1e-2, batch_size=256, seed=SEED, ckpt=None):
    torch.manual_seed(seed); np.random.seed(seed)
    fit_i, es_i = next(GroupShuffleSplit(n_splits=1, test_size=0.05, random_state=seed).split(X, groups=scaffolds))
    Xf, Xe, Yf, Ye = X[fit_i], X[es_i], Y[fit_i], Y[es_i]
    net = make_net(X.shape[1]).to(DEVICE); opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=weight_decay)
    lossf = nn.BCEWithLogitsLoss(); rng = np.random.default_rng(seed); best, best_state, bad = 1e9, None, 0
    for ep in range(max_epochs):
        t0 = time.time(); net.train(); perm = rng.permutation(len(fit_i)); tr_loss = 0.0
        for xb, yb in batches(Xf, Yf, [perm[i:i + batch_size] for i in range(0, len(perm), batch_size)], DEVICE):
            opt.zero_grad(); loss = lossf(net(xb), yb.float()); loss.backward(); opt.step(); tr_loss += loss.item() * len(xb)
        net.eval(); es = 0.0
        with torch.no_grad():
            for xb, yb in batches(Xe, Ye, [np.arange(i, min(i + 1024, Xe.shape[0])) for i in range(0, Xe.shape[0], 1024)], DEVICE):
                es += lossf(net(xb), yb.float()).item() * len(xb)
        es /= Xe.shape[0]; tr_loss /= len(fit_i)
        tick(f"  epoch {ep:2d} | train BCE {tr_loss:.5f} | inner-holdout BCE {es:.5f} | {time.time() - t0:.0f}s")
        if es < best - 1e-5:
            best, bad, best_state = es, 0, {k: v.detach().clone() for k, v in net.state_dict().items()}
            if ckpt: torch.save(best_state, ckpt)
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(best_state); return net

def predict_mlp(net, X, bs=1024):
    net.eval(); out = []
    with torch.no_grad():
        for xb, _ in batches(X, None, [np.arange(i, min(i + bs, X.shape[0])) for i in range(0, X.shape[0], bs)], DEVICE):
            out.append(torch.sigmoid(net(xb)).cpu().numpy())
    return np.vstack(out).astype(np.float32)


In [ ]:
# ======================================== 10b. DEEP MODEL: TRAIN + PREDICT ========================================
DEEP_FILE = f"{WORK}/deep_pred{'_debug' if DEBUG else ''}.npz"
if RUN_DEEP:
    if os.path.exists(DEEP_FILE):
        z = np.load(DEEP_FILE); P_test_deep, P_hold_deep = z["test"], z["hold"]; tick("deep predictions loaded from cache")
    else:
        m = SPLIT_TRAIN
        keep = np.flatnonzero(np.asarray((X_ALL[m] != 0).sum(0)).ravel() > 0)          # columns that occur in the training rows
        Xtr = X_ALL[m][:, keep].astype(np.float32).tocsr()
        tick(f"training the deep MLP on {Xtr.shape[0]:,} spectra x {Xtr.shape[1]:,} columns ({deep[m].key.nunique():,} molecules) on {DEVICE}")
        net = fit_mlp(Xtr, Y_ALL[m], deep.scaffold.values[m].astype(str), DEEP_MAX_EPOCHS, DEEP_PATIENCE, ckpt=f"{WORK}/deep_best{'_debug' if DEBUG else ''}.pt")
        P_test_deep = predict_mlp(net, X_TEST[:, keep].tocsr()); P_hold_deep = predict_mlp(net, X_ALL[SPLIT_HOLD][:, keep].tocsr())
        np.savez(DEEP_FILE, test=P_test_deep, hold=P_hold_deep); del net, Xtr; gc.collect()
        tick("deep MLP fitted")
    holdout_sanity(P_hold_deep, "deep MLP")
    SUB_DEEP, RANKED_DEEP, _ = make_submission(P_test_deep, "deep")
    if FINAL_CHOICE == "deep": publish(SUB_DEEP, "deep MLP")


In [ ]:
# ======================================== 10c. FREEZE: everything the inference-only notebook needs ========================================
# WHY: the hidden-test rerun and every new experiment (prior weight, new pools) would otherwise repeat canonicalisation + features + training.
# Frozen = trained weights, the training-column selection, feature settings, calibration offsets and the TRAINING part of the library (fingerprints
# + masses). COCONUT and the popularity table are NOT copied: the inference notebook reads them from their own datasets.
if FREEZE and RUN_DEEP:
    assert not EXTRA_CANDIDATES_PATH, "EXTRA_CANDIDATES_PATH rows are not frozen; use EXTRA_POOL_DIR"
    import shutil
    fdir = f"{WORK}/frozen"; os.makedirs(fdir, exist_ok=True)
    ckpt = f"{WORK}/deep_best{'_debug' if DEBUG else ''}.pt"
    assert os.path.exists(ckpt), f"{ckpt} missing: the deep model has to be trained (or restored) before it can be frozen"
    keep = np.flatnonzero(np.asarray((X_ALL[SPLIT_TRAIN] != 0).sum(0)).ravel() > 0)              # same selection as the training step
    shutil.copy(ckpt, f"{fdir}/deep_best.pt"); np.save(f"{fdir}/keep.npy", keep)
    pd.DataFrame({"smiles": LIB.smiles[:N_TRAIN_LIB], "key": LIB.keys[:N_TRAIN_LIB], "mass": LIB.mass[:N_TRAIN_LIB]}).to_parquet(f"{fdir}/train_library.parquet")
    np.save(f"{fdir}/train_library_fp.npy", fp_u8[:N_TRAIN_LIB])
    json.dump({"bias": {k: float(v) for k, v in BIAS.items()}, "hw": {k: float(v) for k, v in HW.items()},
               "top_inst": [str(x) for x in FEAT.top_inst], "cols": [str(x) for x in FEAT.cols], "ce_med": float(FEAT.ce_med),
               "n_in": int(len(keep)), "n_cols_full": int(X_ALL.shape[1]), "n_library": int(N_TRAIN_LIB), "debug": bool(DEBUG), "seed": SEED},
              open(f"{fdir}/frozen_meta.json", "w"), indent=1)
    tick("frozen -> " + fdir + ": " + ", ".join(f"{os.path.basename(f)} {os.path.getsize(f) / 1e6:.0f} MB" for f in sorted(glob.glob(f"{fdir}/*"))))


In [ ]:
# ======================================== 11. FINAL FILES + A LOOK AT WHAT WE SUBMITTED ========================================
chosen = {"baseline": globals().get("SUB_BASELINE"), "deep": globals().get("SUB_DEEP")}.get(FINAL_CHOICE)
if chosen is None: chosen = globals().get("SUB_DEEP") if globals().get("SUB_DEEP") is not None else globals().get("SUB_BASELINE")
assert chosen is not None, "no model was run"
publish(chosen, f"the '{FINAL_CHOICE}' model (final)")
print("\nfiles in", WORK); [print(f"  {os.path.basename(f):38s} {os.path.getsize(f) / 1e6:8.1f} MB") for f in sorted(glob.glob(f"{WORK}/*")) if os.path.isfile(f)]

if globals().get("SUB_BASELINE") is not None and globals().get("SUB_DEEP") is not None:
    top1 = lambda s: s.smiles.str.split(";").str[0]
    print(f"top-1 candidate identical between baseline and deep: {(top1(SUB_BASELINE).values == top1(SUB_DEEP).values).mean():.1%} of molecules")
print("\nexample rows:"); print((chosen.assign(smiles=chosen.smiles.str.split(";").str[:3].str.join(";"))).head(5).to_string(index=False))

# PROBE: how many test molecules have a candidate in the library that fits their mass tightly AND whose fingerprint matches the
# prediction almost perfectly? A cosine near 1.0 for many molecules would suggest the test molecules ARE in the library.
keys, Pm, L, H, _ = pool_molecules(P_test_deep if globals().get("SUB_DEEP") is not None else P_test_ridge, test.molecule_id.values, test_lo, test_hi)
ranked, ncand = rank_queries(Pm, L, H, LIB)
best_cos = np.array([float(LIB.fp_unit[r[0]] @ q) for r, q in zip(ranked, Pm)])
print(f"\nPROBE: cosine between the predicted fingerprint and the best candidate: median {np.median(best_cos):.3f}; "
      f"share of molecules with cosine > 0.9: {(best_cos > 0.9).mean():.1%} (for a validation molecule WITH its answer in the library this is typically ~0.6-0.8)")
tick("done")
